In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_df = spark.table("workspace.default.bronze_sales")

display(bronze_df.limit(10))

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0,14.1694
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0,1.9656
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0,34.47


In [0]:
bronze_df.printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
silver_df = bronze_df.dropDuplicates()

print("Rows after removing duplicates:", silver_df.count())

Rows after removing duplicates: 9994


In [0]:
silver_df = silver_df.dropna()

print("Rows after removing nulls:", silver_df.count())

Rows after removing nulls: 9694


In [0]:
from pyspark.sql.functions import col, expr

silver_df = (
    silver_df
    .withColumn("Sales", expr("try_cast(Sales as double)"))
    .withColumn("Profit", expr("try_cast(Profit as double)"))
    .withColumn("Discount", expr("try_cast(Discount as double)"))
    .withColumn("Quantity", expr("try_cast(Quantity as int)"))
)

In [0]:
# Get fresh data from bronze to inspect
bronze_fresh = spark.table("workspace.default.bronze_sales")
display(bronze_fresh.select("Sales", "Profit", "Discount", "Quantity", "Product_Name").limit(30))

Sales,Profit,Discount,Quantity,Product_Name
261.96,41.9136,0,2,Bush Somerset Collection Bookcase
731.94,219.582,0,3,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back"
14.62,6.8714,0,2,Self-Adhesive Address Labels for Typewriters by Universal
957.5775,-383.031,0.45,5,Bretford CR4500 Series Slim Rectangular Table
22.368,2.5164,0.2,2,Eldon Fold 'N Roll Cart System
48.86,14.1694,0,7,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood"
7.28,1.9656,0,4,Newell 322
907.152,90.7152,0.2,6,Mitel 5320 IP Phone VoIP phone
18.504,5.7825,0.2,3,DXL Angle-View Binders with Locking Rings by Samsill
114.9,34.47,0,5,Belkin F5C206VTEL 6 Outlet Surge


In [0]:
from pyspark.sql.functions import current_timestamp

silver_df = silver_df.withColumn(
    "Last_Updated",
    current_timestamp()
)

display(silver_df.limit(10))

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit,Last_Updated
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136,2026-08-05T15:10:58.666Z
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582,2026-08-05T15:10:58.666Z
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714,2026-08-05T15:10:58.666Z
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031,2026-08-05T15:10:58.666Z
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164,2026-08-05T15:10:58.666Z
6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694,2026-08-05T15:10:58.666Z
7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.28,4,0.0,1.9656,2026-08-05T15:10:58.666Z
8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152,2026-08-05T15:10:58.666Z
9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825,2026-08-05T15:10:58.666Z
10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47,2026-08-05T15:10:58.666Z


In [0]:
# Drop existing materialized view if it exists
spark.sql("DROP MATERIALIZED VIEW IF EXISTS default.2_silver")

# Save as Delta table
silver_df.write.mode("overwrite").saveAsTable("default.2_silver")

In [0]:
display(spark.table("default.2_silver").limit(10))

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit,Last_Updated
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164,2026-08-05T15:12:09.460Z
17,CA-2014-105893,2014-11-11,2014-11-18,Standard Class,PK-19075,Pete Kriz,Consumer,United States,Madison,Wisconsin,53711,Central,OFF-ST-10004186,Office Supplies,Storage,"""Stur-D-Stor Shelving, Vertical 5-Shelf: 72""""H x 36""""W x 18 1/2""""D""",665.88,6,0.0,13.3176,2026-08-05T15:12:09.460Z
21,CA-2014-143336,2014-08-27,2014-09-01,Second Class,ZD-21925,Zuschuss Donatelli,Consumer,United States,San Francisco,California,94109,West,OFF-BI-10002215,Office Supplies,Binders,"Wilson Jones Hanging View Binder, White, 1""""",22.72,4,0.2,7.384,2026-08-05T15:12:09.460Z
53,CA-2015-115742,2015-04-18,2015-04-22,Standard Class,DP-13000,Darren Powers,Consumer,United States,New Albany,Indiana,47150,Central,FUR-CH-10003061,Furniture,Chairs,"Global Leather Task Chair, Black",89.99,1,0.0,17.0981,2026-08-05T15:12:09.460Z
118,CA-2015-110457,2015-03-02,2015-03-06,Standard Class,DK-13090,Dave Kipp,Consumer,United States,Seattle,Washington,98103,West,FUR-TA-10001768,Furniture,Tables,Hon Racetrack Conference Tables,787.53,3,0.0,165.3813,2026-08-05T15:12:09.460Z
140,CA-2016-145583,2016-10-13,2016-10-19,Standard Class,LC-16885,Lena Creighton,Consumer,United States,Roseville,California,95661,West,FUR-FU-10001706,Furniture,Furnishings,Longer-Life Soft White Bulbs,43.12,14,0.0,20.6976,2026-08-05T15:12:09.460Z
145,CA-2017-155376,2017-12-22,2017-12-27,Standard Class,SG-20080,Sandra Glassco,Consumer,United States,Independence,Missouri,64055,Central,OFF-AP-10001058,Office Supplies,Appliances,Sanyo 2.5 Cubic Foot Mid-Size Office Refrigerators,839.43,3,0.0,218.2518,2026-08-05T15:12:09.460Z
147,CA-2014-110072,2014-10-22,2014-10-28,Standard Class,MG-17680,Maureen Gastineau,Home Office,United States,Newark,Ohio,43055,East,FUR-FU-10000521,Furniture,Furnishings,"""Seth Thomas 14"""" Putty-Colored Wall Clock""",93.888,4,0.2,12.9096,2026-08-05T15:12:09.460Z
186,CA-2016-105018,2016-11-28,2016-12-02,Standard Class,SK-19990,Sally Knutson,Consumer,United States,Fairfield,Connecticut,6824,East,OFF-BI-10001890,Office Supplies,Binders,Avery Poly Binder Pockets,7.16,2,0.0,3.4368,2026-08-05T15:12:09.460Z
200,US-2017-124303,2017-07-06,2017-07-13,Standard Class,FH-14365,Fred Hopkins,Corporate,United States,Philadelphia,Pennsylvania,19120,East,OFF-PA-10002749,Office Supplies,Paper,"Wirebound Message Books, 5-1/2 x 4 Forms, 2 or 4 Forms per Page",16.056,3,0.2,5.8203,2026-08-05T15:12:09.460Z


In [0]:
spark.table("default.2_silver").printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)

